<a href="https://colab.research.google.com/github/lygitdata/GarmentIQ/blob/main/test/adv_usage_classification_model_fine_tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Advanced Usage - GarmentIQ Classification Model Fine-tuning

Fine-tuning adapts the pretrained GarmentIQ classifier to your own catalog without
training from scratch. It starts from the shipped tinyViT weights, freezes the backbone,
and retrains only the classifier head, so it needs far less data and time than full
training.

This tutorial shows how to prepare a fine-tuning dataset, freeze and unfreeze the right
layers, run cross-validated fine-tuning, and evaluate the resulting model.

## Table of Contents

1. [Prerequisites](#prerequisites)
2. [Prepare the fine-tuning data](#data)
3. [Fine-tune the model](#finetune)
4. [Evaluate the fine-tuned model](#evaluate)

<a name="prerequisites"></a>
## Prerequisites

Install the package, then download the fine-tuning dataset and the base model. On Colab
you can keep this section collapsed.

> **Your data must be a zip file with the same structure as ours**, that is an image
> folder plus a `metadata.csv` naming each file and its label. See
> [the example dataset](https://www.kaggle.com/datasets/lygitdata/zara-clothes-image-data)
> for the exact layout.

In [ ]:
# @title Install GarmentIQ
!pip install garmentiq -q

In [ ]:
# @title Import GarmentIQ and choose a device

import torch
import torch.optim as optim

import garmentiq as giq
from garmentiq.classification.model_definition import tinyViT
from garmentiq.classification.utils import CachedDataset

# GarmentIQ never grabs an accelerator on its own. For training and fine-tuning the
# device is passed inside `param`, and it defaults to "cpu". Training on CPU is slow,
# so use a GPU runtime ("cuda") or Apple Silicon ("mps") where available.
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

print("Using device:", device)

In [ ]:
# @title Download the fine-tuning data and the base model

# About 2.2 GB
!curl -sL -o zara-clothes-image-data.zip \
  https://www.kaggle.com/api/v1/datasets/download/lygitdata/zara-clothes-image-data

# The pretrained tinyViT that will be fine-tuned
!mkdir -p ./models
!wget -q -O ./models/tiny_vit.pt \
    https://huggingface.co/lygitdata/garmentiq/resolve/main/tiny_vit.pt

print("Downloads finished.")

<a name="data"></a>
## Prepare the fine-tuning data

`train_test_split` unpacks the archive and splits it. This dataset is small and every
image is wanted for fine-tuning, so `test_size=0` keeps the test set empty. The summary
below confirms its size is 0.

In [ ]:
data = giq.classification.train_test_split(
    output_dir="data",
    train_zip_dir="zara-clothes-image-data.zip",
    metadata_csv="metadata.csv",
    label_column="garment",
    test_size=0,
    verbose=True,
)

Caching the images in memory avoids re-reading them from disk on every
epoch, which is usually the bottleneck. The normalization statistics must match the ones
the base model was trained with.

In [ ]:
train_images, train_labels, _ = giq.classification.load_data(
    df=data["train_metadata"],
    img_dir=data["train_images"],
    label_column="garment",
    resize_dim=(120, 184),
    normalize_mean=[0.8047, 0.7808, 0.7769],
    normalize_std=[0.2957, 0.3077, 0.3081],
)

<a name="finetune"></a>
## Fine-tune the model

The fine-tuning specific keys live in `param`:

| Key | Purpose |
|---|---|
| `pretrained_path` | the weights to start from |
| `freeze_layers` | freeze the backbone so its features are preserved |
| `unfreeze_patterns` | substrings of layer names to keep trainable |
| `device` | where to run, defaults to `"cpu"` |

Freezing everything except the classifier head is what makes fine-tuning cheap: the
backbone already knows how to read garment images, and only the final mapping onto your
labels is relearned. A small learning rate keeps the pretrained features intact.

Training is cross-validated, and the model with the lowest cross-entropy is saved as the
best one. Five folds and five epochs are used here for demonstration.

In [ ]:
giq.classification.fine_tune_pytorch_nn(
    model_class=tinyViT,
    model_args={"num_classes": 9, "img_size": (120, 184), "patch_size": 6},
    dataset_class=CachedDataset,
    dataset_args={
        "metadata_df": data["train_metadata"],
        "raw_labels": data["train_metadata"]["garment"],
        "cached_images": train_images,
        "cached_labels": train_labels,
    },
    param={
        "pretrained_path": "./models/tiny_vit.pt",
        "freeze_layers": True,
        "unfreeze_patterns": ["classifier", "fc"],
        "optimizer_class": optim.AdamW,
        "optimizer_args": {"lr": 0.00002, "weight_decay": 1e-4},
        "n_fold": 5,
        "n_epoch": 5,
        "patience": 2,
        "batch_size": 128,
        "model_save_dir": "finetuned_models",
        "best_model_name": "best_finetuned.pt",
        "device": device,
    },
)

<a name="evaluate"></a>
## Evaluate the fine-tuned model

`test_pytorch_nn` loads the weights itself, so it takes a model path rather than a loaded
model. Its `device` also goes inside `param`.

> Scores here are measured on the fine-tuning data itself, so they show how well the model
> fitted it, not how well it generalizes. Hold out a test set to measure that.

In [ ]:
giq.classification.test_pytorch_nn(
    model_path="finetuned_models/best_finetuned.pt",
    model_class=tinyViT,
    model_args={"num_classes": 9, "img_size": (120, 184), "patch_size": 6},
    dataset_class=CachedDataset,
    dataset_args={
        "raw_labels": data["train_metadata"]["garment"],
        "cached_images": train_images,
        "cached_labels": train_labels,
    },
    param={"batch_size": 64, "device": device},
)

To use the fine-tuned model for prediction, load it exactly like the
shipped one. See the
[classification tutorial](https://colab.research.google.com/github/lygitdata/GarmentIQ/blob/main/test/tutorial_classification.ipynb).